In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Makes plots Look clean
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [4]:
deliveries = pd.read_csv("./Data/deliveries.csv")
matches = pd.read_csv("./Data/matches.csv")
most_runs = pd.read_csv("./Data/most_runs_average_strikerate.csv")
players = pd.read_excel("./Data/Players.xlsx")
teams = pd.read_csv("./Data/teams.csv")
home_away = pd.read_csv("./Data/teamwise_home_and_away.csv")

In [ ]:
# Exploring Files

# Exploring Matches
print(matches.shape)    # rows and columns
print(matches.columns.tolist())
print(matches.head(3))
print(matches.isnull().sum())

In [ ]:
# Exploring Deliveries
print(deliveries.shape)
print(deliveries.columns.tolist())
print(deliveries.head(3))

In [ ]:
for name, df in [("most_runs", most_runs), ("players", players),
                 ("teams", teams), ("home_away", home_away)]:
  print(f"\n {name.upper()}")
  print(df.shape)
  print(df.columns.tolist())
  print(df.head(3))

#### Ananlysis 1: Team Performance

In [ ]:
# Most Match Wins by Team
team_wins = matches['winner'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=team_wins.values, y=team_wins.index, palette='viridis')
plt.title('Top 10 Teams by Total Match Wins', fontsize=16)
plt.xlabel('Number of Wins')
plt.ylabel('Team')
plt.tight_layout()
plt.savefig('team_wins.png')   # saves chart as image
plt.show()

# Toss Decision Analysis
toss_decision = matches['toss_decision'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(toss_decision.values, labels=toss_decision.index,
        autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
plt.title('Toss Decision: Bat vs Field', fontsize=14)
plt.savefig('toss_decision.png')
plt.show()

# Does Toss Winner Win the Match?
matches['toss_match_winner'] = matches['toss_winner'] == matches['winner']
toss_impact = matches['toss_match_winner'].value_counts()

print("Toss winner also won the match:")
print(toss_impact)

plt.figure(figsize=(6, 5))
sns.countplot(x='toss_match_winner', data=matches)
plt.title('Does Toss Winner Win the Match?')
plt.xticks([0, 1], ['No', 'Yes'])
plt.xlabel('')
plt.savefig('toss_impact.png')
plt.show()

#### Analysis 2: Home Vs Away Performance

In [ ]:
print(home_away.head())
print(home_away.columns.tolist())

# Plot home wins vs away wins per team
plt.figure(figsize=(14, 7))

# Adjust column names based on what you see from head() above
# Common column names could be: Team, Home, Away
home_away_sorted = home_away.sort_values('home_wins', ascending=False)  # ✅ correct column name

x = np.arange(len(home_away_sorted))
width = 0.35

plt.figure(figsize=(14, 7))
plt.bar(x - width/2, home_away_sorted['home_wins'], width, label='Home Wins', color='steelblue')
plt.bar(x + width/2, home_away_sorted['away_wins'], width, label='Away Wins', color='coral')

plt.xticks(x, home_away_sorted['team'], rotation=45, ha='right')
plt.title('Home vs Away Wins by Team', fontsize=16)
plt.ylabel('Number of Wins')
plt.legend()
plt.tight_layout()
plt.savefig('home_away.png')
plt.show()

#### Analysis 3: Batting Stats (from most_runs)

In [ ]:
print(most_runs.head())
print(most_runs.columns.tolist())

In [ ]:
# Top 10 Run Scorers
top_batsmen = most_runs.sort_values('total_runs', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='total_runs', y='batsman', data=top_batsmen, palette='magma')
plt.title('Top 10 Run Scorers in IPL', fontsize=16)
plt.xlabel('Total Runs')
plt.ylabel('Batsman')
plt.tight_layout()
plt.savefig('top_batsmen.png')
plt.show()

# Top 10 by Strike Rate (min 500 runs)
qualified = most_runs[most_runs['total_runs'] >= 500]
top_sr = qualified.sort_values('strikerate', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='strikerate', y='batsman', data=top_sr, palette='rocket')
plt.title('Top 10 Strike Rates in IPL (min 500 runs)', fontsize=16)
plt.xlabel('Strike Rate')
plt.ylabel('Batsman')
plt.tight_layout()
plt.savefig('top_strikerate.png')
plt.show()

# Top 10 by Batting Average (min 500 runs)
top_avg = qualified.sort_values('average', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='average', y='batsman', data=top_avg, palette='coolwarm')
plt.title('Top 10 Batting Averages in IPL (min 500 runs)', fontsize=16)
plt.xlabel('Batting Average')
plt.ylabel('Batsman')
plt.tight_layout()
plt.savefig('top_average.png')
plt.show()

#### Analysis 4: Bowling Stats (from deliveries)

In [ ]:
# Top 10 Wicket Takers
# Filter only valid dismissals (exclude run outs)
wickets = deliveries[deliveries['dismissal_kind'].notna()]
wickets = wickets[wickets['dismissal_kind'] != 'run out']

top_bowlers = wickets['bowler'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_bowlers.values, y=top_bowlers.index, palette='flare')
plt.title('Top 10 Wicket Takers in IPL', fontsize=16)
plt.xlabel('Total Wickets')
plt.savefig('top_bowlers.png')
plt.show()

# Most Economical Bowlers 
bowler_runs = deliveries.groupby('bowler')['total_runs'].sum()
bowler_balls = deliveries.groupby('bowler')['ball'].count()
bowler_overs = bowler_balls / 6
economy = (bowler_runs / bowler_overs).reset_index()
economy.columns = ['bowler', 'economy']

# Filter bowlers with at least 300 balls
qualified_bowlers = bowler_balls[bowler_balls >= 300].index
economy = economy[economy['bowler'].isin(qualified_bowlers)]
top_economy = economy.sort_values('economy').head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='economy', y='bowler', data=top_economy, palette='Blues_r')
plt.title('Top 10 Most Economical Bowlers (min 300 balls)', fontsize=16)
plt.xlabel('Economy Rate')
plt.savefig('top_economy.png')
plt.show()

#### Analysis 5: Match Trends Over Seasons

In [ ]:
print(matches.columns.tolist())
print(matches.head(3))

In [ ]:
# Matches Played Per Season 
matches_per_season = matches['Season'].value_counts().sort_index()

plt.figure(figsize=(12, 5))
sns.lineplot(x=matches_per_season.index, y=matches_per_season.values, marker='o', color='steelblue')
plt.title('Number of Matches Per Season', fontsize=16)
plt.xlabel('Season')
plt.ylabel('Matches Played')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('matches_per_season.png')
plt.show()

# Wins Per Team Per Season 
season_wins = matches.groupby(['Season', 'winner']).size().reset_index(name='wins')
top_teams = matches['winner'].value_counts().head(5).index  # top 5 teams only

plt.figure(figsize=(14, 6))
for team in top_teams:
    team_data = season_wins[season_wins['winner'] == team]
    plt.plot(team_data['Season'], team_data['wins'], marker='o', label=team)

plt.title('Wins Per Season - Top 5 Teams', fontsize=16)
plt.xlabel('Season')
plt.ylabel('Number of Wins')
plt.xticks(rotation=45)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('season_wins_trend.png')
plt.show()

## Key Insights

1. **Mumbai Indians** have the most IPL wins across all seasons.
2. Teams prefer to **field after winning the toss** ~60% of the time.
3. Toss winning does **not significantly** decide the match outcome.
4. **Virat Kohli** is the all-time leading run scorer in IPL.
5. Teams perform noticeably better at **home venues** than away.
6. Average target scores have **increased over the seasons**, 
   showing the game is becoming more batting-friendly.
